<a href="https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Prioritise pages for refresh when they combine meaningful search visibility with content staleness. Search volume is weighted more strongly because the signal showed the clearest directional separation in the audit, while staleness is used as a supporting signal because its relationship with decline was mixed but the flag-linked test provided directional support. Raw CTR is excluded from the baseline because its distribution contains extreme values and its relationship with position was inconsistent.

Actions:
- REFRESH: strong refresh opportunity
- REVIEW: moderate opportunity for human review
- MONITOR: lower-priority page

Reason codes identify the combination of volume and staleness that produced the recommendation.

In [9]:
# Section 1 — Load dataset and define the baseline rule

import pandas as pd
import numpy as np

# Same 30,000-row dataset used in the signal audit
DATA_URL = "https://raw.githubusercontent.com/thahsinj06/Fly-rank-ml-internship-work/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)

# ------------------------------------------------------------
# Baseline rule
#
# Prioritise pages using two signals supported by our audit:
# 1. Search volume — stronger signal
# 2. Content staleness — supporting signal
#
# CTR is intentionally excluded because its distribution
# contained extreme values and its pattern was mixed.
# ------------------------------------------------------------

# Convert both signals to percentile scores.
# This prevents impressions from dominating simply because
# its raw numerical scale is much larger.
df["_volume_score"] = df["impressions_90d"].rank(pct=True)

# Cap staleness at 180 days so extreme stale pages do not
# receive unlimited additional weight.
df["_staleness_score"] = (
    df["days_since_last_update"].clip(upper=180) / 180
)

# Weighted baseline score
df["_baseline_score"] = (
    0.60 * df["_volume_score"] +
    0.40 * df["_staleness_score"]
)

# Action labels
df["_action"] = np.select(
    [
        df["_baseline_score"] >= 0.70,
        df["_baseline_score"] >= 0.45
    ],
    [
        "REFRESH",
        "REVIEW"
    ],
    default="MONITOR"
)

# Reason codes
df["_reason_code"] = np.select(
    [
        (df["_volume_score"] >= 0.75) &
        (df["_staleness_score"] >= 0.50),

        (df["_volume_score"] >= 0.75) &
        (df["_staleness_score"] < 0.50),

        (df["_volume_score"] < 0.75) &
        (df["_staleness_score"] >= 0.50)
    ],
    [
        "HIGH_VOLUME_STALE",
        "HIGH_VOLUME_FRESH",
        "LOWER_VOLUME_STALE"
    ],
    default="LOWER_VOLUME_FRESH"
)

print("\nBaseline rule created successfully.")

print("\nDataset check:")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nActions:")
print(df["_action"].value_counts())

print("\nReason codes:")
print(df["_reason_code"].value_counts())

print("\nScore summary:")
print(df["_baseline_score"].describe())

Dataset shape: (30000, 44)

Baseline rule created successfully.

Dataset check:
Rows: 30000
Columns: 49

Actions:
_action
MONITOR    17069
REVIEW     10079
REFRESH     2852
Name: count, dtype: int64

Reason codes:
_reason_code
LOWER_VOLUME_FRESH    16360
LOWER_VOLUME_STALE     6139
HIGH_VOLUME_FRESH      4295
HIGH_VOLUME_STALE      3206
Name: count, dtype: int64

Score summary:
count    30000.000000
mean         0.401875
std          0.212576
min          0.012982
25%          0.222904
50%          0.405980
75%          0.567711
max          0.991960
Name: _baseline_score, dtype: float64


## 2. Build the ranked queue

The baseline ranks pages by a transparent weighted score combining search-volume percentile and capped staleness. Higher scores indicate stronger observed refresh opportunities. The queue is decision-support only and is not a prediction of future performance.



In [10]:
# Section 2 — Build ranked queue and write the required CSV

# Rank all pages from highest to lowest baseline score
df["_rank"] = (
    df["_baseline_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Build the required output
baseline_queue = df.copy()

baseline_queue = baseline_queue.sort_values("_rank")

# Keep useful identifying/context columns while avoiding
# temporary intermediate columns in the final queue.
output_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "_baseline_score",
    "_rank",
    "_reason_code",
    "_action"
]

# Keep only columns that actually exist in this dataset
output_columns = [col for col in output_columns if col in baseline_queue.columns]

baseline_queue = baseline_queue[output_columns]

# Rename internal fields to clean output names
baseline_queue = baseline_queue.rename(
    columns={
        "_baseline_score": "score",
        "_rank": "rank",
        "_reason_code": "reason_code",
        "_action": "action"
    }
)

# Create output directory and write CSV
import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(output_path, index=False)

print(f"Queue written to: {output_path}")
print(f"Rows written: {len(baseline_queue):,}")
print(f"Columns written: {len(baseline_queue.columns)}")

print("\nTop 10:")
display(baseline_queue.head(10))

Queue written to: work/outputs/baseline_action_score.csv
Rows written: 30,000
Columns written: 10

Top 10:


,content_type,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,score,rank,reason_code,action
16751,keyword article,231,194,61678,19.7,0.15,0.991960,1,HIGH_VOLUME_STALE,REFRESH
16514,keyword article,231,194,59472,24.8,0.13,0.991720,2,HIGH_VOLUME_STALE,REFRESH
7021,keyword article,231,194,25715,22.2,0.23,0.973360,3,HIGH_VOLUME_STALE,REFRESH
21268,keyword article,231,193,13299,10.5,0.49,0.944860,4,HIGH_VOLUME_STALE,REFRESH
11489,keyword article,231,194,7812,39.0,0.01,0.911700,5,HIGH_VOLUME_STALE,REFRESH
12045,keyword article,231,193,7558,17.9,0.20,0.909320,6,HIGH_VOLUME_STALE,REFRESH
8006,keyword article,306,151,21272,12.6,2.45,0.902256,7,HIGH_VOLUME_STALE,REFRESH
698,keyword article,231,194,4590,31.0,0.00,0.870370,8,HIGH_VOLUME_STALE,REFRESH
5327,keyword article,231,194,4556,16.4,0.33,0.869710,9,HIGH_VOLUME_STALE,REFRESH
26810,keyword article,231,194,4429,25.3,0.38,0.867070,10,HIGH_VOLUME_STALE,REFRESH


## 3. Top-20 review

The top 20 pages are reviewed as individual decision-support recommendations. For each page, the review records the assigned action and reason code, explains why the rule selected it, and identifies a condition that could make the recommendation wrong. The purpose is to challenge the baseline rather than assume that a high score is automatically correct.

In [11]:
# Section 3 — Generate the Top-20 review

top20 = baseline_queue.head(20).copy()

top20["confidence_note"] = np.where(
    top20["action"] == "REFRESH",
    "Strong score from high visibility + staleness",
    "Moderate baseline priority; human review needed"
)

top20["what_would_make_it_wrong"] = (
    "High impressions may not represent valuable or refreshable traffic; "
    "staleness alone does not prove content needs updating."
)

review_columns = [
    "rank",
    "content_type",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "score",
    "reason_code",
    "action",
    "confidence_note",
    "what_would_make_it_wrong"
]

review_columns = [
    col for col in review_columns
    if col in top20.columns
]

display(top20[review_columns])

,rank,content_type,days_since_last_update,impressions_90d,avg_position,score,reason_code,action,confidence_note,what_would_make_it_wrong
16751,1,keyword article,194,61678,19.7,0.991960,HIGH_VOLUME_STALE,REFRESH,Strong score from high visibility + staleness,High impressions may not represent valuable or...
16514,2,keyword article,194,59472,24.8,0.991720,HIGH_VOLUME_STALE,REFRESH,Strong score from high visibility + staleness,High impressions may not represent valuable or...
7021,3,keyword article,194,25715,22.2,0.973360,HIGH_VOLUME_STALE,REFRESH,Strong score from high visibility + staleness,High impressions may not represent valuable or...
21268,4,keyword article,193,13299,10.5,0.944860,HIGH_VOLUME_STALE,REFRESH,Strong score from high visibility + staleness,High impressions may not represent valuable or...
11489,5,keyword article,194,7812,39.0,0.911700,HIGH_VOLUME_STALE,REFRESH,Strong score from high visibility + staleness,High impressions may not represent valuable or...
12045,6,keyword article,193,7558,17.9,0.909320,HIGH_VOLUME_STALE,REFRESH,Strong score from high visibility + staleness,High impressions may not represent valuable or...
8006,7,keyword article,151,21272,12.6,0.902256,HIGH_VOLUME_STALE,REFRESH,Strong score from high visibility + staleness,High impressions may not represent valuable or...
698,8,keyword article,194,4590,31.0,0.870370,HIGH_VOLUME_STALE,REFRESH,Strong score from high visibility + staleness,High impressions may not represent valuable or...
5327,9,keyword article,194,4556,16.4,0.869710,HIGH_VOLUME_STALE,REFRESH,Strong score from high visibility + staleness,High impressions may not represent valuable or...
26810,10,keyword article,194,4429,25.3,0.867070,HIGH_VOLUME_STALE,REFRESH,Strong score from high visibility + staleness,High impressions may not represent valuable or...


## 4. Weak picks + leakage check

The strongest baseline recommendations are concentrated in the high-volume, stale group. This is reasonable given the rule design, but it may over-prioritise pages where one or both signals are extreme. The weakest-looking picks are therefore reviewed for cases where high search volume or staleness may not translate into a genuine refresh opportunity.

The baseline uses only information available in the current dataset and does not use the observed decline outcome, future-window performance, or product flags as scoring inputs..*

In [12]:
# Section 4 — Weak picks + leakage check

# ------------------------------------------------------------
# Weak-pick review
# Find high-ranked pages where the two scoring signals disagree.
# These are useful candidates for skeptical review.
# ------------------------------------------------------------

weak_pick_candidates = df[
    (
        (df["_volume_score"] >= 0.75) &
        (df["_staleness_score"] < 0.25)
    )
    |
    (
        (df["_volume_score"] < 0.25) &
        (df["_staleness_score"] >= 0.75)
    )
].copy()

weak_pick_candidates = weak_pick_candidates.sort_values(
    "_baseline_score",
    ascending=False
).head(10)

print("=" * 60)
print("WEAK-PICK CANDIDATES")
print("=" * 60)

weak_columns = [
    "content_type",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "_volume_score",
    "_staleness_score",
    "_baseline_score",
    "_reason_code",
    "_action"
]

weak_columns = [
    col for col in weak_columns
    if col in weak_pick_candidates.columns
]

display(weak_pick_candidates[weak_columns])


# ------------------------------------------------------------
# Leakage check
# Confirm that the scoring formula only uses the intended
# pre-existing signals.
# ------------------------------------------------------------

scoring_inputs = {
    "impressions_90d",
    "days_since_last_update"
}

print("\n" + "=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

print("Scoring inputs:")
for col in sorted(scoring_inputs):
    print(f"✓ {col}")

forbidden_inputs = [
    "trend_direction",
    "is_declining_label"
]

print("\nOutcome/label fields excluded from scoring:")

for col in forbidden_inputs:
    print(f"✓ {col} excluded")

print("\nFinal scoring formula:")
print("score = 0.60 × volume percentile + 0.40 × capped staleness")

print("\nNo future-window performance variable is used in the score.")


WEAK-PICK CANDIDATES


,content_type,days_since_last_update,impressions_90d,avg_position,ctr,_volume_score,_staleness_score,_baseline_score,_reason_code,_action
8399,keyword article,40,46866,4.6,0.03,0.981100,0.222222,0.677549,HIGH_VOLUME_FRESH,REVIEW
6237,keyword article,34,98124,4.7,0.42,0.994200,0.188889,0.672076,HIGH_VOLUME_FRESH,REVIEW
28858,keyword article,35,47962,7.4,0.25,0.981733,0.194444,0.666818,HIGH_VOLUME_FRESH,REVIEW
25441,keyword article,41,24155,3.2,0.05,0.952567,0.227778,0.662651,HIGH_VOLUME_FRESH,REVIEW
15839,keyword article,34,40913,8.0,0.09,0.977300,0.188889,0.661936,HIGH_VOLUME_FRESH,REVIEW
8931,keyword article,40,22668,5.0,0.04,0.949233,0.222222,0.658429,HIGH_VOLUME_FRESH,REVIEW
10559,keyword article,26,131617,7.4,0.26,0.997133,0.144444,0.656058,HIGH_VOLUME_FRESH,REVIEW
29713,keyword article,34,31756,6.4,0.07,0.966767,0.188889,0.655616,HIGH_VOLUME_FRESH,REVIEW
19499,keyword article,25,173450,22.6,0.04,0.998567,0.138889,0.654696,HIGH_VOLUME_FRESH,REVIEW
15969,keyword article,26,103793,16.1,0.47,0.994567,0.144444,0.654518,HIGH_VOLUME_FRESH,REVIEW



LEAKAGE CHECK
Scoring inputs:
✓ days_since_last_update
✓ impressions_90d

Outcome/label fields excluded from scoring:
✓ trend_direction excluded
✓ is_declining_label excluded

Final scoring formula:
score = 0.60 × volume percentile + 0.40 × capped staleness

No future-window performance variable is used in the score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.